In [ ]:
from process_eeg import get_psd_df, get_psd_avg_df, get_osc_df
from visualization.vis_eeg import compare_epo_psd, compare_band_metric
from utils.gen_utils import get_sids
import numpy as np

%load_ext autoreload
%autoreload 2

In [ ]:
test = False
show = True
save = True

# 0. Overview of all epoch-types
## PSD

In [ ]:
# Load dataframe with power spectra
psd_df = get_psd_df(test=test, log=True)
psd_df

In [ ]:
# Load dataframe with average power spectra
psd_avg_df = get_psd_avg_df(log=True)
psd_avg_df

In [ ]:
# Average across subjects
# psd_avg_df = psd_avg_df[psd_avg_df['cond'] != 'cTBS']
compare_epo_psd(psd_avg_df, 'epo_type', show=show, save=save, plot_subj='average')

In [ ]:
# For each subject separately  
psd_df_lin = get_psd_df(test=test, log=False)  # need linear psd dfs to compute std across blocks
for sid in get_sids(include_04=True):
    df = psd_df_lin[psd_df_lin['sid'] == sid]
    df['pw_avg'] = np.log10(df['pw_avg'])
    plot_df = df.groupby(['cond', 'epo_type', 'freq'], as_index=False).agg(  # average together blocks of the same condition
            **{
                "sid": ("sid", 'first'),
                "freq": ("freq", 'first'),
                "pw_avg": ('pw_avg', 'mean'),
                "pw_std": ('pw_avg', 'std'),
                "N": ('pw_avg', 'count'),
            }
        )
    compare_epo_psd(plot_df, 'epo_type', show=show, save=False, plot_subj=sid)  # sem_n is number of blocks by condition

## Oscillations

In [ ]:
# Load dataframe with power spectra
osc_df = get_osc_df(test=test)
# osc_df = osc_df[osc_df['cond'] != 'cTBS']
osc_df 

In [ ]:
osc_df_no4 = osc_df[~(osc_df['sid'] == '04')]

## Power in canonical bands

In [ ]:
compare_band_metric(osc_df_no4, super_col='epo_type', show=show, save=save, metric_name='abs_pw')
compare_band_metric(osc_df_no4, super_col='epo_type', show=show, save=save, metric_name='rel_pw')

In [ ]:
# For each subject separately
for sid in get_sids(test, include_04=True):
    df = osc_df[osc_df['sid'] == sid]
    compare_band_metric(df, super_col='epo_type', show=show, save=save, metric_name='abs_pw')
    compare_band_metric(df, super_col='epo_type', show=show, save=save, metric_name='rel_pw')

## SNR 
Based on oscillations / background noise

In [ ]:
# Average across patients
compare_band_metric(osc_df_no4, super_col='epo_type', show=show, save=save, metric_name='osc_snr')

In [ ]:
# For each subject separately
for sid in get_sids(test):
    df = osc_df[osc_df['sid'] == sid]
    compare_band_metric(df, super_col='epo_type', show=show, save=save, metric_name='osc_snr')